In [1]:
import torch
import os
from torch.utils.cpp_extension import load_inline

In [2]:
os.makedirs("./load_inline_cuda", exist_ok=True)

In [14]:
cud='''
__global__ void relu(const float* inn,float* out,int size){
    int idx=blockIdx.x*blockDim.x+threadIdx.x;
    if(idx<size){
        float val =inn[idx];
        out[idx]=val>0?val:0;
    }
}
torch::Tensor relu(torch::Tensor input){
    int size=input.numel();
    auto output=torch::empty_like(input);
    int bts=256;
    int nb=(size+bts-1)/bts;
    relu<<<nb,bts>>>(input.data_ptr<float>(),output.data_ptr<float>(),size);
    return output;
}
    '''

In [15]:
cpps='torch::Tensor relu(torch::Tensor input);'

In [16]:
extension = load_inline(
    name="relu_extension",
    cpp_sources=cpps,
    cuda_sources=cud,
    functions=["relu"],
    with_cuda=True,
    build_directory="./load_inline_cuda"
)

In [17]:
x = torch.randn(10, device="cuda")
y = extension.relu(x)

In [18]:
x

tensor([-2.4625, -0.4786, -0.4604, -0.5138,  0.8537, -0.4736,  0.6353, -0.4376,
         0.4732,  0.2204], device='cuda:0')

In [19]:
y

tensor([0.0000, 0.0000, 0.0000, 0.0000, 0.8537, 0.0000, 0.6353, 0.0000, 0.4732,
        0.2204], device='cuda:0')